In [5]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
import pandas as pd
from src.data.loader import load_raw_data

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score


In [3]:
df = load_raw_data("../data/raw/db.csv")

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['month'] = df['Date'].dt.month
df['year'] = df['Date'].dt.year
df['day'] = df['Date'].dt.day
df_expenses = df[df['Type'] == 'Expenses'].copy()
gasto_10d = df_expenses[df_expenses['day'] <= 10].groupby(['year', 'month'])['Amount'].sum().reset_index()
gasto_total = df_expenses.groupby(['year', 'month'])['Amount'].sum().reset_index()
df_ml = pd.merge(gasto_10d, gasto_total, on=['year', 'month'], suffixes=('_10d', '_total'))

X = df_ml[['Amount_10d']] 
y = df_ml['Amount_total']
df_ml.head()

,year,month,Amount_10d,Amount_total
0,2010,2,16.62,77.85
1,2010,3,86.17,288.42
2,2010,4,15.51,133.17
3,2010,5,93.51,168.51
4,2010,6,57.14,266.54


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo A: Árbol de Decisión (Más simple)
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)

# Modelo B: Random Forest (Más robusto, recomendado por Kaushik et al.)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predicciones
y_pred_tree = tree_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

In [7]:
print(f"--- RESULTADOS ÁRBOL DE DECISIÓN ---")
print(f"R2: {r2_score(y_test, y_pred_tree):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_tree):.2f}€")

print(f"\n--- RESULTADOS RANDOM FOREST ---")
print(f"R2: {r2_score(y_test, y_pred_rf):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_rf):.2f}€")

--- RESULTADOS ÁRBOL DE DECISIÓN ---
R2: -0.0805
MAE: 100.54€

--- RESULTADOS RANDOM FOREST ---
R2: 0.0146
MAE: 96.58€
